# CURE-Rec advanced reviewer ablations
Run one guarded action at a time. Results are new evidence, not replacements for archived studies.


In [1]:
from pathlib import Path
import sys, pandas as pd
CWD=Path.cwd().resolve()
CANDIDATES=[CWD, CWD/'paper-ideas'/'CURE-Rec'/'code', *CWD.parents]
ROOT=next((p for p in CANDIDATES if (p/'pyproject.toml').exists() and (p/'cure_rec').exists()), None)
if ROOT is None: raise RuntimeError('Open from the CURE-Rec code/notebooks directory or repository root.')
sys.path[:] = [str(ROOT), *[x for x in sys.path if x != str(ROOT)]]
for name in list(sys.modules):
    if name == 'cure_rec' or name.startswith('cure_rec.'):
        del sys.modules[name]
import importlib
importlib.invalidate_caches()
from cure_rec.config import load_settings
from cure_rec.pipeline import run_experiment
from cure_rec.revision_advanced import select_objective, sampled_shapley
FULL_CONFIG=ROOT/'configs'/'curesim_full.yaml'


In [ ]:
RUN_OBJECTIVE_ABLATION=False
RUN_SAMPLED_SHAPLEY=False
SEED=300
assert not (RUN_OBJECTIVE_ABLATION and RUN_SAMPLED_SHAPLEY)


## Action 1 — maximin/mean and hard/penalty ablation


In [ ]:
if RUN_OBJECTIVE_ABLATION:
    cfg=load_settings(FULL_CONFIG); cfg.run.seed=SEED; cfg.run.output_root=ROOT/'runs'
    _,game,_=run_experiment(cfg)
    rows=[select_objective(game,cfg,objective=o,constraint_mode=c) for o in ('maximin','mean') for c in ('hard','penalty')]
    display(pd.DataFrame(rows))
else: print('Disabled.')


## Action 2 — exact versus sampled Shapley


In [ ]:
if RUN_SAMPLED_SHAPLEY:
    cfg=load_settings(FULL_CONFIG); cfg.run.seed=SEED; cfg.run.output_root=ROOT/'runs'
    _,game,_=run_experiment(cfg)
    exact=game.robust_shapley
    rows=[]
    for budget in (32,128,512,2048):
        est=sampled_shapley(game.robust_improvements,budget,seed=SEED)
        for player in exact: rows.append({'permutations':budget,'intervention':player,'exact':exact[player],'estimate':est[player],'absolute_error':abs(exact[player]-est[player])})
    display(pd.DataFrame(rows))
else: print('Disabled.')


CRN-off, scaling, user-level bootstrap, and a second dataset require additional simulator/evaluator implementations and should not be substituted with unvalidated shortcuts.


## Action 3 — CRN on versus independent-shock variance
This runs the same exact game for five seeds under matched common random numbers and independent coalition shock streams.


In [2]:
RUN_OBJECTIVE_ABLATION = False
RUN_SAMPLED_SHAPLEY = False
RUN_CRN_VARIANCE = True

CRN_SEEDS = (300, 301, 302, 303, 304)
if RUN_CRN_VARIANCE:
    rows=[]
    for common in (True, False):
        for seed in CRN_SEEDS:
            cfg=load_settings(FULL_CONFIG); cfg.run.seed=seed; cfg.run.common_random_numbers=common; cfg.simulator.click_feedback_weight=1.0; cfg.run.output_root=ROOT/'runs'
            _,game,decision=run_experiment(cfg)
            rows.append({'seed':seed,'common_random_numbers':common,'selected_mask':decision.selected_mask,'selected_interventions':';'.join(decision.selected_interventions),'robust_lower_improvement':decision.lower_improvement,'base_feasible':decision.base_feasible,'click_feedback_weight':cfg.simulator.click_feedback_weight})
    crn_results=pd.DataFrame(rows)
    print(crn_results.groupby('common_random_numbers',as_index=False).agg(mean=('robust_lower_improvement','mean'),sd=('robust_lower_improvement','std')))
    display(crn_results)
else: print('CRN variance study disabled.')


2026-08-11 09:04:17,692 | INFO | run_started | {"config_hash": "20ff5323c6add1a5", "run_id": "curesim-full-20260811T080417Z-732696d3"}
2026-08-11 09:04:17,693 | INFO | exact_game_started | {}
2026-08-11 09:04:17,695 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "nominal"}
2026-08-11 09:10:08,397 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.1406521512316774, "scenario": "nominal", "shapley_efficiency_gap": 0.0}
2026-08-11 09:10:08,400 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "fatigue_stress"}
2026-08-11 09:14:55,071 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.12824569278118791, "scenario": "fatigue_stress", "shapley_efficiency_gap": 0.0}
2026-08-11 09:14:55,073 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "popularity_stress"}
2026-08-11 09:18:40,094 | INFO | scenario_game_completed | {"grand_coalition_improvem

,seed,common_random_numbers,selected_mask,selected_interventions,robust_lower_improvement,base_feasible
0,300,True,1,repeat_cap,0.289936,False
1,301,True,1,repeat_cap,0.295858,True
2,302,True,1,repeat_cap,0.294127,True
3,303,True,1,repeat_cap,0.292217,False
4,304,True,1,repeat_cap,0.292591,False
5,300,False,1,repeat_cap,0.289936,False
6,301,False,1,repeat_cap,0.295858,True
7,302,False,1,repeat_cap,0.294127,True
8,303,False,1,repeat_cap,0.292217,False
9,304,False,1,repeat_cap,0.292591,False
